In [1]:
# This notebook is a replication of Gu, Kelly, Xiu (2020) Emprical Asset Pricing using Machine Learning;
# 3-hidden-layer neural networks with Adam for Stochastic gradient descent are used to predict 35 years of US stock returns (1987-2021);
# Replication period is 1987-2016; I extended the results to 2021;
# Parameters choice follows the paper but with 5 random seeds (the paper uses 10) due to limited resources;
# Predictive accuracy can potentially be improved with more seeds;
# At each epoch, validation loss and testing loss are printed for tracking/ debugging purposes;
# The model is chosen based on minimum validation loss, NOT testing loss;
# Pooled out-of-sample R-squared can be found at the very end.

import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import gc, time

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else torch.device('cpu')
print(f'device: {device}, version {torch.__version__}')
print(f'-----------------------------------------------------------------------------------------------------------------------------------------------------------')


# Design matrix and target variable
## Design matrix: 94 characteristics, 752 interaction terms, 74 one-hot encoded processed and saved from "1_Preprocessing"
Xdata = np.memmap('Xdata.dat', dtype='float32', mode='c', shape=(4096791, 920))
print(f'X shape: {Xdata.shape}, with characteristics ranked and mapped to [-1,1]')
print(f'X structure (last row as e.g):')
print(f'94 characteristics:')
print(Xdata[-1,:94])
print(f'752 interactions:')
print(Xdata[-1,94:752])
print(f'74 sic dummies')
print(Xdata[-1,-74:])
print(f'-----------------------------------------------------------------------------------------------------------------------------------------------------------')
## Target is monthly excess returns, processed and saved from "1_Preprocessing"
print('Excess returns')
ydata = np.memmap('ydata.dat', dtype='float32', mode='c', shape=(4096791, 1))
print(f'y shape: {ydata.shape}')
print(f'y structure: {ydata[:5]}')

# Sample split: 35 splits, processed and saved from "1_Preprocessing"
print(f'-----------------------------------------------------------------------------------------------------------------------------------------------------------')
idx = pd.read_csv('idx.csv')
print(f'35 sample splits:')
print(f'e.g: first split has (i) training subsample: ydata[:478009], Xdata[:478009], (ii) validation subsample: ydata[478009:1248579], Xdata[478009:1248579], (iii) testing subsample: ydata[1248579:1331524], Xdata[1248579:1331524], equivalent to prediction for the year 1987')
print(idx)
idx = idx.to_numpy()
print(f'-----------------------------------------------------------------------------------------------------------------------------------------------------------')

# Testing sample excess returns
## For replication period 1987-2016
y_true_rep = []
for i in range(30):
    y_true_rep.append(torch.from_numpy(ydata[idx[i][1]:idx[i][2]]))
y_true_rep = torch.cat(y_true_rep, dim=0)
print(f'Replication period observations (1987-2016): {y_true_rep.shape[0]}')

## For extended period 1987-2021
y_true = []
for i in range(35):
    y_true.append(torch.from_numpy(ydata[idx[i][1]:idx[i][2]]))
y_true = torch.cat(y_true, dim=0)
print(f'Extended period observations (1987-2021): {y_true.shape[0]}')
print(f'-----------------------------------------------------------------------------------------------------------------------------------------------------------')

# Indices of the top and bottom 1000 stocks in testing sample by market value; saved from 1_Preprocessing
## Extended period
t, b = np.load('top1000.npy'), np.load('bot1000.npy')
t, b = list(t), list(b)
## Replication period
t_rep, b_rep = [], []
for i in range(len(t)):
    if t[i] < y_true_rep.shape[0]:
        t_rep.append(t[i])
for i in range(len(b)):
    if b[i] < y_true_rep.shape[0]:
        b_rep.append(b[i])

device: cuda, version 2.8.0+cu129
-----------------------------------------------------------------------------------------------------------------------------------------------------------
X shape: (4096791, 920), with characteristics ranked and mapped to [-1,1]
X structure (last row as e.g):
94 characteristics:
[ 0.99913144  0.652948    0.65090483  0.9924099   1.          0.55195564
  0.15792567  0.7106109   0.99123317  0.8727193   1.         -0.75718665
  0.8365736   0.7575669  -0.77412313 -0.6725559  -0.89726424 -0.91824263
  0.5219668   0.66302925  0.94383    -0.7071751  -0.56745744  0.899869
  1.          0.97700816  0.86321306 -0.83675253  1.         -0.48370197
 -0.68169916  0.          0.         -0.9985448   0.97322077 -0.5462087
  0.66962594  0.76207095  0.87990695  0.72723323  0.96798605  0.95082206
 -0.9391646   0.5243778   1.          0.6071584  -0.7465853   0.85427576
  0.87107104 -0.67588013  0.9129929   0.89378726  0.85095876 -0.72475994
  0.88343024  0.8364323  -0.913

In [18]:
# Neural network with 3 hidden layers-----------------------------------------------------------------------------------------------------------------------
class NN3(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(920, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 8)
        self.out = nn.Linear(8, 1)
        self.bn1 = nn.BatchNorm1d(32)
        self.bn2 = nn.BatchNorm1d(16)

    def forward(self, x):
        x = self.fc1(x)  #1st: 920 -> 32
        x = F.relu(x)
        x = self.bn1(x)
        x = self.fc2(x)  #2nd: 32 -> 16
        x = F.relu(x)
        x = self.bn2(x)
        x = self.fc3(x)  #3re: 16 -> 8
        x = F.relu(x)
        x = self.out(x)  #out: 8 -> 1
        return x

#Parameters initialization----------------------------------------------------------------------------------------------------------------------------------
def _model_init(model,seed):
    torch.manual_seed(seed)
    for n, p in model.named_parameters():
        if n.startswith('fc') and n.endswith('weight'):
            nn.init.kaiming_normal_(p)
        elif n.startswith('bn') and n.endswith('weight'):
            nn.init.constant_(p, 1)
        elif n.endswith('bias'):
            nn.init.zeros_(p)

#Training, validation, testing------------------------------------------------------------------------------------------------------------------------------
def train_loop(Xtrain, ytrain, Xval, yval, Xtest, ytest, reg_param, learningrate, n_batchsize, model, max_epochs,
               n_patience, seed):
    Xtrain = torch.from_numpy(Xtrain)
    ytrain = torch.from_numpy(ytrain)
    Xval = torch.from_numpy(Xval).to(device)
    yval = torch.from_numpy(yval).to(device)
    Xtest = torch.from_numpy(Xtest).to(device)
    ytest = torch.from_numpy(ytest).to(device)
    trainset = TensorDataset(Xtrain, ytrain)
    trainloader = DataLoader(trainset, batch_size=n_batchsize, shuffle=True)
    linear_weights = [p for n, p in model.named_parameters() if n.startswith('fc') and n.endswith('weight')]

    minvalloss = float('inf')
    for r in reg_param:
        for lr in learningrate:
            _model_init(model=model,seed=seed)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0)
            epoch_loss = []
            for e in range(max_epochs):
                model.train()
                for batch_idx, (X, y) in enumerate(trainloader):
                    xb = X.to(device)
                    yb = y.to(device)
                    pred_tr = model(xb)
                    l1_penalty = sum(p.abs().sum() for p in linear_weights)
                    loss_tr = (yb - pred_tr).pow(2).mean() + r * l1_penalty
                    #Backpropagation
                    loss_tr.backward()  #compute gradient
                    optimizer.step()  #update model parameters
                    optimizer.zero_grad()  #reset gradients to zeros
                model.eval()
                with torch.no_grad():
                    pred_v = model(Xval)
                    loss_v = (yval - pred_v).pow(2).mean()
                    epoch_loss.append(loss_v)
                    #print estimation results; printing test loss only for tracking; the model is chosen based on min validation loss
                    print(f'lambda: {r}, learning rate: {lr}, epoch: {e}, val loss: {loss_v:.6f}, '
                          f'test loss: {(ytest - model(Xtest)).pow(2).sum():.6f}'
                          )
                    #get out-of-sample prediction using min validation loss model
                    if loss_v.item() < minvalloss:
                        minvalloss = loss_v.item()
                        model_dict = model.state_dict()
                        pred_t = model(Xtest)
                        sse = (ytest - pred_t).pow(2).sum()
                        best_lambda = r
                        best_lr = lr
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
                gc.collect()
                time.sleep(30)
                if e - torch.argmin(torch.tensor(epoch_loss)) == n_patience:
                    break
    return minvalloss, best_lambda, best_lr, model_dict, pred_t, sse

#Parameter grid-----------------------------------------------------------------------------------------------------------------------------------------
model_NN3 = NN3().to(device)
pred_NN3, sse_NN3 = [], []
reg_param = [0.00001, 0.0001, 0.001]
learning_rate = [0.01]
seed = [555, 666, 777, 888, 999]

In [18]:
#NN3_split_0_generate prediction for 1987
pred_0 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[0][0]],
        ytrain=ydata[:idx[0][0]],
        Xval=Xdata[idx[0][0]:idx[0][1]],
        yval=ydata[idx[0][0]:idx[0][1]],
        Xtest=Xdata[idx[0][1]:idx[0][2]],
        ytest=ydata[idx[0][1]:idx[0][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_0: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_0.append(pred_t)
pred_NN3.append(sum(pred_0)/len(seed))
del pred_0
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.028430, test loss: 3110.427246
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.027638, test loss: 3043.033691
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.027472, test loss: 3026.584473
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.027484, test loss: 3018.054199
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.027819, test loss: 3022.032959
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.028577, test loss: 3057.911621
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.028622, test loss: 3071.219727
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.029042, test loss: 3091.135254
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.026101, test loss: 3003.903320
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.026972, test loss: 2989.763184
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.026675, test loss: 2995.515137
lambda: 

0

In [20]:
#NN3_split_1_generate prediction for 1988
pred_1 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[1][0]],
        ytrain=ydata[:idx[1][0]],
        Xval=Xdata[idx[1][0]:idx[1][1]],
        yval=ydata[idx[1][0]:idx[1][1]],
        Xtest=Xdata[idx[1][1]:idx[1][2]],
        ytest=ydata[idx[1][1]:idx[1][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_1: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_1.append(pred_t)
pred_NN3.append(sum(pred_1)/len(seed))
del pred_1
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.026290, test loss: 2226.257080
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.026752, test loss: 2277.748779
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.026825, test loss: 2288.276855
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.027082, test loss: 2314.228027
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.027407, test loss: 2352.411865
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.026833, test loss: 2309.332031
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.026266, test loss: 2220.377441
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.026466, test loss: 2239.272949
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.026446, test loss: 2245.193115
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.026345, test loss: 2236.595459
lambda: 0.0001, learning rate: 0.01, epoch: 4, val loss: 0.026526, test loss: 2252.370605
lambda

0

In [63]:
#NN3_split_2_generate prediction for 1989
pred_2 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[2][0]],
        ytrain=ydata[:idx[2][0]],
        Xval=Xdata[idx[2][0]:idx[2][1]],
        yval=ydata[idx[2][0]:idx[2][1]],
        Xtest=Xdata[idx[2][1]:idx[2][2]],
        ytest=ydata[idx[2][1]:idx[2][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_2: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_2.append(pred_t)
pred_NN3.append(sum(pred_2)/len(seed))
del pred_2
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.026487, test loss: 2145.043701
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.026617, test loss: 2167.938965
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.026570, test loss: 2160.328613
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.026597, test loss: 2149.931885
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.026639, test loss: 2152.413330
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.026540, test loss: 2148.194336
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.026457, test loss: 2141.114746
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.026514, test loss: 2150.371338
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.026736, test loss: 2175.981201
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.026516, test loss: 2155.409180
lambda: 0.0001, learning rate: 0.01, epoch: 4, val loss: 0.026679, test loss: 2182.123047
lambda

57

In [74]:
#NN3_split_3_generate prediction for 1990
pred_3 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[3][0]],
        ytrain=ydata[:idx[3][0]],
        Xval=Xdata[idx[3][0]:idx[3][1]],
        yval=ydata[idx[3][0]:idx[3][1]],
        Xtest=Xdata[idx[3][1]:idx[3][2]],
        ytest=ydata[idx[3][1]:idx[3][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_3: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_3.append(pred_t)
pred_NN3.append(sum(pred_3)/len(seed))
del pred_3
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.027246, test loss: 3021.633301
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.027314, test loss: 3054.637695
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.027295, test loss: 3029.174072
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.027421, test loss: 3040.733887
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.027236, test loss: 3008.981445
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.027294, test loss: 3014.713379
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.027280, test loss: 2993.473877
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.027236, test loss: 3007.782959
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.027227, test loss: 3017.371582
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.027272, test loss: 3020.799561
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.027280, test loss: 2992.032471
lambda: 1e

76

In [76]:
#NN3_split_4_generate prediction for 1991
pred_4 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[4][0]],
        ytrain=ydata[:idx[4][0]],
        Xval=Xdata[idx[4][0]:idx[4][1]],
        yval=ydata[idx[4][0]:idx[4][1]],
        Xtest=Xdata[idx[4][1]:idx[4][2]],
        ytest=ydata[idx[4][1]:idx[4][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_4: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_4.append(pred_t)
pred_NN3.append(sum(pred_4)/len(seed))
del pred_4
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.028507, test loss: 4222.601562
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.028749, test loss: 4212.869141
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.028590, test loss: 4222.142578
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.028583, test loss: 4223.818848
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.028515, test loss: 4232.011719
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.028830, test loss: 4221.430664
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.028512, test loss: 4209.373047
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.028990, test loss: 4204.039062
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.028692, test loss: 4209.320312
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.028613, test loss: 4207.462891
lambda: 0.0001, learning rate: 0.01, epoch: 4, val loss: 0.028919, test loss: 4183.096680
lambda

95

In [78]:
#NN3_split_5_generate prediction for 1992
pred_5 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[5][0]],
        ytrain=ydata[:idx[5][0]],
        Xval=Xdata[idx[5][0]:idx[5][1]],
        yval=ydata[idx[5][0]:idx[5][1]],
        Xtest=Xdata[idx[5][1]:idx[5][2]],
        ytest=ydata[idx[5][1]:idx[5][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_5: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_5.append(pred_t)
pred_NN3.append(sum(pred_5)/len(seed))
del pred_5
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.031370, test loss: 3949.984375
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.031441, test loss: 3950.647461
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.031259, test loss: 3946.673096
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.032228, test loss: 3958.120605
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.031604, test loss: 3928.451172
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.032044, test loss: 3938.468506
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.031315, test loss: 3929.111328
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.031862, test loss: 3990.450195
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.032235, test loss: 3960.374756
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.033604, test loss: 4257.399902
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.031958, test loss: 4104.803711
lambda: 

114

In [80]:
#NN3_split_6_generate prediction for 1993
pred_6 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[6][0]],
        ytrain=ydata[:idx[6][0]],
        Xval=Xdata[idx[6][0]:idx[6][1]],
        yval=ydata[idx[6][0]:idx[6][1]],
        Xtest=Xdata[idx[6][1]:idx[6][2]],
        ytest=ydata[idx[6][1]:idx[6][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_6: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_6.append(pred_t)
pred_NN3.append(sum(pred_6)/len(seed))
del pred_6
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.033056, test loss: 2893.162598
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.033131, test loss: 2892.194824
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.033169, test loss: 2890.750732
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.033311, test loss: 2878.913330
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.033063, test loss: 2876.540771
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.033096, test loss: 2893.168457
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.033043, test loss: 2878.943848
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.033124, test loss: 2879.256348
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.033014, test loss: 2882.493164
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.033005, test loss: 2893.021240
lambda: 0.0001, learning rate: 0.01, epoch: 4, val loss: 0.033080, test loss: 2880.197510
lambda

133

In [82]:
#NN3_split_7_generate prediction for 1994
pred_7 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[7][0]],
        ytrain=ydata[:idx[7][0]],
        Xval=Xdata[idx[7][0]:idx[7][1]],
        yval=ydata[idx[7][0]:idx[7][1]],
        Xtest=Xdata[idx[7][1]:idx[7][2]],
        ytest=ydata[idx[7][1]:idx[7][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_7: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_7.append(pred_t)
pred_NN3.append(sum(pred_7)/len(seed))
del pred_7
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.033838, test loss: 2206.114746
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.033980, test loss: 2210.498535
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.034066, test loss: 2208.504395
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.034694, test loss: 2272.336426
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.034422, test loss: 2230.626953
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.034295, test loss: 2219.229492
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.033769, test loss: 2199.530273
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.033995, test loss: 2192.630615
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.034133, test loss: 2259.199707
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.034660, test loss: 2325.051758
lambda: 0.0001, learning rate: 0.01, epoch: 4, val loss: 0.034033, test loss: 2265.636719
lambda

152

In [84]:
#NN3_split_8
pred_8 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[8][0]],
        ytrain=ydata[:idx[8][0]],
        Xval=Xdata[idx[8][0]:idx[8][1]],
        yval=ydata[idx[8][0]:idx[8][1]],
        Xtest=Xdata[idx[8][1]:idx[8][2]],
        ytest=ydata[idx[8][1]:idx[8][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_8: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_8.append(pred_t)
pred_NN3.append(sum(pred_8)/len(seed))
del pred_8
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.033034, test loss: 2653.584961
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.033069, test loss: 2653.958496
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.033193, test loss: 2671.121094
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.033228, test loss: 2668.539062
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.033292, test loss: 2676.660645
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.033482, test loss: 2677.033691
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.033148, test loss: 2654.379395
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.033227, test loss: 2655.631836
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.033306, test loss: 2673.218750
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.033173, test loss: 2666.312744
lambda: 0.0001, learning rate: 0.01, epoch: 4, val loss: 0.033673, test loss: 2720.676025
lambda

171

In [86]:
#NN3_split_9
pred_9 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[9][0]],
        ytrain=ydata[:idx[9][0]],
        Xval=Xdata[idx[9][0]:idx[9][1]],
        yval=ydata[idx[9][0]:idx[9][1]],
        Xtest=Xdata[idx[9][1]:idx[9][2]],
        ytest=ydata[idx[9][1]:idx[9][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_9: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_9.append(pred_t)
pred_NN3.append(sum(pred_9)/len(seed))
del pred_9
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.032727, test loss: 3039.605713
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.032636, test loss: 3032.831299
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.032508, test loss: 3023.225586
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.032607, test loss: 3030.363037
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.032525, test loss: 3020.804688
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.032542, test loss: 3031.660400
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.032618, test loss: 3039.980469
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.032743, test loss: 3036.462891
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.032914, test loss: 3044.081543
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.032622, test loss: 3038.287842
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.032563, test loss: 3027.789062
lambda: 

190

In [88]:
#NN3_split_10
pred_10 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[10][0]],
        ytrain=ydata[:idx[10][0]],
        Xval=Xdata[idx[10][0]:idx[10][1]],
        yval=ydata[idx[10][0]:idx[10][1]],
        Xtest=Xdata[idx[10][1]:idx[10][2]],
        ytest=ydata[idx[10][1]:idx[10][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_10: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_10.append(pred_t)
pred_NN3.append(sum(pred_10)/len(seed))
del pred_10
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.032866, test loss: 3165.949707
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.032958, test loss: 3183.425049
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.033008, test loss: 3206.338135
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.033263, test loss: 3232.531494
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.032970, test loss: 3184.206055
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.033021, test loss: 3203.480713
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.032955, test loss: 3171.316406
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.032985, test loss: 3188.280273
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.033338, test loss: 3219.404297
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.033052, test loss: 3180.173828
lambda: 0.0001, learning rate: 0.01, epoch: 4, val loss: 0.032947, test loss: 3174.041992
lambda

209

In [90]:
#NN3_split_11
pred_11 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[11][0]],
        ytrain=ydata[:idx[11][0]],
        Xval=Xdata[idx[11][0]:idx[11][1]],
        yval=ydata[idx[11][0]:idx[11][1]],
        Xtest=Xdata[idx[11][1]:idx[11][2]],
        ytest=ydata[idx[11][1]:idx[11][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_11: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_11.append(pred_t)
pred_NN3.append(sum(pred_11)/len(seed))
del pred_11
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.032939, test loss: 5122.219727
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.032990, test loss: 5109.037109
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.033042, test loss: 5105.945312
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.032827, test loss: 5104.072754
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.032913, test loss: 5100.967773
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.032926, test loss: 5097.013184
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.033067, test loss: 5100.146973
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.032895, test loss: 5114.035645
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.032931, test loss: 5113.112305
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.032919, test loss: 5120.970703
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.032956, test loss: 5112.295898
lambda: 0

228

In [92]:
#NN3_split_12
pred_12 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[12][0]],
        ytrain=ydata[:idx[12][0]],
        Xval=Xdata[idx[12][0]:idx[12][1]],
        yval=ydata[idx[12][0]:idx[12][1]],
        Xtest=Xdata[idx[12][1]:idx[12][2]],
        ytest=ydata[idx[12][1]:idx[12][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_12: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_12.append(pred_t)
pred_NN3.append(sum(pred_12)/len(seed))
del pred_12
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.034426, test loss: 5798.060547
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.034431, test loss: 5768.133301
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.034399, test loss: 5770.179688
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.034502, test loss: 5800.222656
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.034429, test loss: 5715.867188
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.034410, test loss: 5757.594727
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.034375, test loss: 5761.363770
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.034413, test loss: 5758.726562
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.034365, test loss: 5742.360840
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.034440, test loss: 5758.524414
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.034387, test loss: 5785.689453
lambda: 1e

247

In [94]:
#NN3_split_13
pred_13 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[13][0]],
        ytrain=ydata[:idx[13][0]],
        Xval=Xdata[idx[13][0]:idx[13][1]],
        yval=ydata[idx[13][0]:idx[13][1]],
        Xtest=Xdata[idx[13][1]:idx[13][2]],
        ytest=ydata[idx[13][1]:idx[13][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_13: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_13.append(pred_t)
pred_NN3.append(sum(pred_13)/len(seed))
del pred_13
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.036939, test loss: 6125.819824
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.036702, test loss: 6144.960449
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.037130, test loss: 6163.233887
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.036870, test loss: 6150.047852
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.036730, test loss: 6157.790039
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.036782, test loss: 6128.641602
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.036749, test loss: 6165.995605
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.036943, test loss: 6201.177246
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.036504, test loss: 6253.592773
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.036644, test loss: 6158.653320
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.036557, test loss: 6214.408691
lambda:

266

In [96]:
#NN3_split_14
pred_14 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[14][0]],
        ytrain=ydata[:idx[14][0]],
        Xval=Xdata[idx[14][0]:idx[14][1]],
        yval=ydata[idx[14][0]:idx[14][1]],
        Xtest=Xdata[idx[14][1]:idx[14][2]],
        ytest=ydata[idx[14][1]:idx[14][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_14: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_14.append(pred_t)
pred_NN3.append(sum(pred_14)/len(seed))
del pred_14
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.039490, test loss: 6057.656738
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.039866, test loss: 6154.698242
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.039412, test loss: 6023.784180
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.039440, test loss: 6035.822754
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.039684, test loss: 6160.553711
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.039584, test loss: 6121.916016
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.039745, test loss: 6142.126953
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.039421, test loss: 6089.541016
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.040303, test loss: 6166.605957
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.039605, test loss: 6078.905273
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.039529, test loss: 6032.573242
lambda: 

285

In [98]:
#NN3_split_15
pred_15 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[15][0]],
        ytrain=ydata[:idx[15][0]],
        Xval=Xdata[idx[15][0]:idx[15][1]],
        yval=ydata[idx[15][0]:idx[15][1]],
        Xtest=Xdata[idx[15][1]:idx[15][2]],
        ytest=ydata[idx[15][1]:idx[15][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_15: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_15.append(pred_t)
pred_NN3.append(sum(pred_15)/len(seed))
del pred_15
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.042626, test loss: 3466.170898
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.042558, test loss: 3464.167480
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.042522, test loss: 3451.305420
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.042468, test loss: 3455.885986
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.042542, test loss: 3453.172852
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.042558, test loss: 3439.702637
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.042403, test loss: 3456.359131
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.042516, test loss: 3445.176514
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.042599, test loss: 3445.577393
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.042447, test loss: 3457.857422
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.042505, test loss: 3442.093262
lambda: 1e

304

In [100]:
#NN3_split_16
pred_16 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[16][0]],
        ytrain=ydata[:idx[16][0]],
        Xval=Xdata[idx[16][0]:idx[16][1]],
        yval=ydata[idx[16][0]:idx[16][1]],
        Xtest=Xdata[idx[16][1]:idx[16][2]],
        ytest=ydata[idx[16][1]:idx[16][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_16: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_16.append(pred_t)
pred_NN3.append(sum(pred_16)/len(seed))
del pred_16
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.042828, test loss: 2711.362793
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.042824, test loss: 2752.388184
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.042865, test loss: 2741.884521
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.042930, test loss: 2727.273926
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.042864, test loss: 2764.176758
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.042761, test loss: 2713.685791
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.042913, test loss: 2759.885254
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.042954, test loss: 2745.431885
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.042848, test loss: 2763.024902
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.043054, test loss: 2785.329102
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.042818, test loss: 2748.504150
lambda: 0.

0

In [103]:
#NN3_split_17
pred_17 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[17][0]],
        ytrain=ydata[:idx[17][0]],
        Xval=Xdata[idx[17][0]:idx[17][1]],
        yval=ydata[idx[17][0]:idx[17][1]],
        Xtest=Xdata[idx[17][1]:idx[17][2]],
        ytest=ydata[idx[17][1]:idx[17][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_17: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_17.append(pred_t)
pred_NN3.append(sum(pred_17)/len(seed))
del pred_17
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.041316, test loss: 1634.895752
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.041192, test loss: 1629.692139
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.041379, test loss: 1648.257812
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.041215, test loss: 1636.987061
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.041192, test loss: 1652.896606
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.041144, test loss: 1628.567139
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.041189, test loss: 1649.652466
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.041306, test loss: 1643.254150
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.041184, test loss: 1643.869385
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.041403, test loss: 1673.801880
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.041221, test loss: 1639.203003
lambda: 0.

342

In [105]:
#NN3_split_18
pred_18 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[18][0]],
        ytrain=ydata[:idx[18][0]],
        Xval=Xdata[idx[18][0]:idx[18][1]],
        yval=ydata[idx[18][0]:idx[18][1]],
        Xtest=Xdata[idx[18][1]:idx[18][2]],
        ytest=ydata[idx[18][1]:idx[18][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_18: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_18.append(pred_t)
pred_NN3.append(sum(pred_18)/len(seed))
del pred_18
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.039299, test loss: 1225.800659
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.039284, test loss: 1233.617432
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.039388, test loss: 1288.258789
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.039410, test loss: 1265.419067
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.039329, test loss: 1261.287842
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.039261, test loss: 1245.090820
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.039317, test loss: 1254.953125
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.039208, test loss: 1232.842407
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.039366, test loss: 1285.030029
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.039239, test loss: 1250.532104
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.039282, test loss: 1252.732178
lambda: 1e

361

In [107]:
#NN3_split_19
pred_19 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[19][0]],
        ytrain=ydata[:idx[19][0]],
        Xval=Xdata[idx[19][0]:idx[19][1]],
        yval=ydata[idx[19][0]:idx[19][1]],
        Xtest=Xdata[idx[19][1]:idx[19][2]],
        ytest=ydata[idx[19][1]:idx[19][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_19: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_19.append(pred_t)
pred_NN3.append(sum(pred_19)/len(seed))
del pred_19
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.037964, test loss: 1194.869873
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.037966, test loss: 1186.844116
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.038031, test loss: 1210.525879
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.037964, test loss: 1198.889771
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.038077, test loss: 1219.354492
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.037913, test loss: 1197.052856
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.037992, test loss: 1207.556152
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.037966, test loss: 1207.670654
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.038712, test loss: 1264.929443
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.038065, test loss: 1218.090820
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.038039, test loss: 1208.363403
lambda: 0.

380

In [109]:
#NN3_split_20
pred_20 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[20][0]],
        ytrain=ydata[:idx[20][0]],
        Xval=Xdata[idx[20][0]:idx[20][1]],
        yval=ydata[idx[20][0]:idx[20][1]],
        Xtest=Xdata[idx[20][1]:idx[20][2]],
        ytest=ydata[idx[20][1]:idx[20][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_20: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_20.append(pred_t)
pred_NN3.append(sum(pred_20)/len(seed))
del pred_20
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.037604, test loss: 1257.553589
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.037660, test loss: 1261.412354
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.037616, test loss: 1258.664551
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.037703, test loss: 1246.880981
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.037675, test loss: 1300.995361
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.037595, test loss: 1246.882324
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.037628, test loss: 1266.703369
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.037693, test loss: 1286.507690
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.037650, test loss: 1282.213623
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.037709, test loss: 1243.986084
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.037694, test loss: 1252.223999
lambda: 0.

399

In [111]:
#NN3_split_21
pred_21 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[21][0]],
        ytrain=ydata[:idx[21][0]],
        Xval=Xdata[idx[21][0]:idx[21][1]],
        yval=ydata[idx[21][0]:idx[21][1]],
        Xtest=Xdata[idx[21][1]:idx[21][2]],
        ytest=ydata[idx[21][1]:idx[21][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_21: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_21.append(pred_t)
pred_NN3.append(sum(pred_21)/len(seed))
del pred_21
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.037343, test loss: 2957.145996
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.037298, test loss: 3036.805420
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.037203, test loss: 2918.816895
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.037970, test loss: 3116.214600
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.037024, test loss: 3058.543213
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.036921, test loss: 3058.399170
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.036979, test loss: 2931.772949
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.036989, test loss: 2964.074219
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.036911, test loss: 2990.139160
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.037113, test loss: 3020.888184
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.036985, test loss: 2950.134033
lambda: 1e

418

In [113]:
#NN3_split_22
pred_22 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[22][0]],
        ytrain=ydata[:idx[22][0]],
        Xval=Xdata[idx[22][0]:idx[22][1]],
        yval=ydata[idx[22][0]:idx[22][1]],
        Xtest=Xdata[idx[22][1]:idx[22][2]],
        ytest=ydata[idx[22][1]:idx[22][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_22: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_22.append(pred_t)
pred_NN3.append(sum(pred_22)/len(seed))
del pred_22
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.037794, test loss: 4373.014648
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.037826, test loss: 4335.092773
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.037752, test loss: 4332.833008
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.037996, test loss: 4386.323730
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.037795, test loss: 4331.830566
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.037818, test loss: 4301.949219
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.037775, test loss: 4321.128418
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.037746, test loss: 4353.617188
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.037796, test loss: 4306.687012
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.037883, test loss: 4293.956543
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.037816, test loss: 4324.587891
lambda: 1e

437

In [115]:
#NN3_split_23
pred_23 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[23][0]],
        ytrain=ydata[:idx[23][0]],
        Xval=Xdata[idx[23][0]:idx[23][1]],
        yval=ydata[idx[23][0]:idx[23][1]],
        Xtest=Xdata[idx[23][1]:idx[23][2]],
        ytest=ydata[idx[23][1]:idx[23][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_23: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_23.append(pred_t)
pred_NN3.append(sum(pred_23)/len(seed))
del pred_23
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.040272, test loss: 1493.729736
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.040314, test loss: 1490.498291
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.040317, test loss: 1497.394043
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.040543, test loss: 1506.771240
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.040279, test loss: 1485.172974
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.040227, test loss: 1501.160645
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.040269, test loss: 1498.081299
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.040277, test loss: 1509.847534
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.040309, test loss: 1513.783081
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.040228, test loss: 1493.478760
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.040327, test loss: 1488.874268
lambda: 0.

456

In [117]:
#NN3_split_24
pred_24 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[24][0]],
        ytrain=ydata[:idx[24][0]],
        Xval=Xdata[idx[24][0]:idx[24][1]],
        yval=ydata[idx[24][0]:idx[24][1]],
        Xtest=Xdata[idx[24][1]:idx[24][2]],
        ytest=ydata[idx[24][1]:idx[24][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_24: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_24.append(pred_t)
pred_NN3.append(sum(pred_24)/len(seed))
del pred_24
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.038235, test loss: 1316.669922
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.038265, test loss: 1309.234131
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.038265, test loss: 1309.935303
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.038205, test loss: 1314.031738
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.038204, test loss: 1301.603760
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.038231, test loss: 1295.807617
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.038195, test loss: 1304.398682
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.038243, test loss: 1311.220825
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.038227, test loss: 1312.855469
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.038292, test loss: 1316.660400
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.038262, test loss: 1314.603760
lambda: 1e

475

In [119]:
#NN3_split_25
pred_25 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[25][0]],
        ytrain=ydata[:idx[25][0]],
        Xval=Xdata[idx[25][0]:idx[25][1]],
        yval=ydata[idx[25][0]:idx[25][1]],
        Xtest=Xdata[idx[25][1]:idx[25][2]],
        ytest=ydata[idx[25][1]:idx[25][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_25: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_25.append(pred_t)
pred_NN3.append(sum(pred_25)/len(seed))
del pred_25
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.035205, test loss: 1234.057617
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.035358, test loss: 1260.258789
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.035165, test loss: 1248.942139
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.034986, test loss: 1225.263672
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.034991, test loss: 1232.681396
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.035308, test loss: 1260.716064
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.035190, test loss: 1234.844727
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.035018, test loss: 1235.850586
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.035358, test loss: 1274.514160
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.037545, test loss: 1319.422119
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.036560, test loss: 1320.509644
lambda: 0

494

In [121]:
#NN3_split_26
pred_26 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[26][0]],
        ytrain=ydata[:idx[26][0]],
        Xval=Xdata[idx[26][0]:idx[26][1]],
        yval=ydata[idx[26][0]:idx[26][1]],
        Xtest=Xdata[idx[26][1]:idx[26][2]],
        ytest=ydata[idx[26][1]:idx[26][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_26: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_26.append(pred_t)
pred_NN3.append(sum(pred_26)/len(seed))
del pred_26
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.030925, test loss: 1083.071777
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.030834, test loss: 1070.822144
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.030804, test loss: 1073.923706
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.030783, test loss: 1069.342896
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.030939, test loss: 1082.633545
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.030946, test loss: 1079.408203
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.031026, test loss: 1094.628540
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.030804, test loss: 1075.397095
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.030812, test loss: 1086.479980
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.030923, test loss: 1082.268066
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.030934, test loss: 1072.548584
lambda: 0

18

In [123]:
#NN3_split_27
pred_27 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[27][0]],
        ytrain=ydata[:idx[27][0]],
        Xval=Xdata[idx[27][0]:idx[27][1]],
        yval=ydata[idx[27][0]:idx[27][1]],
        Xtest=Xdata[idx[27][1]:idx[27][2]],
        ytest=ydata[idx[27][1]:idx[27][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_27: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_27.append(pred_t)
pred_NN3.append(sum(pred_27)/len(seed))
del pred_27
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.026287, test loss: 1098.111572
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.026283, test loss: 1100.175781
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.026326, test loss: 1100.631104
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.026316, test loss: 1102.649536
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.026416, test loss: 1123.889282
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.027180, test loss: 1194.072876
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.026273, test loss: 1105.224121
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.026317, test loss: 1087.941772
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.026363, test loss: 1118.744141
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.026416, test loss: 1125.538452
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.026392, test loss: 1119.583984
lambda: 1e

37

In [125]:
#NN3_split_28
pred_28 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[28][0]],
        ytrain=ydata[:idx[28][0]],
        Xval=Xdata[idx[28][0]:idx[28][1]],
        yval=ydata[idx[28][0]:idx[28][1]],
        Xtest=Xdata[idx[28][1]:idx[28][2]],
        ytest=ydata[idx[28][1]:idx[28][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_28: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_28.append(pred_t)
pred_NN3.append(sum(pred_28)/len(seed))
del pred_28
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.024288, test loss: 1829.069458
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.024369, test loss: 1818.540283
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.024288, test loss: 1811.247681
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.024214, test loss: 1807.604736
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.024316, test loss: 1814.815308
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.024162, test loss: 1816.107666
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.024232, test loss: 1822.037476
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.024274, test loss: 1824.929443
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.024216, test loss: 1810.931152
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.024223, test loss: 1818.044067
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.024266, test loss: 1823.912109
lambda: 0.

56

In [127]:
#NN3_split_29
pred_29 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[29][0]],
        ytrain=ydata[:idx[29][0]],
        Xval=Xdata[idx[29][0]:idx[29][1]],
        yval=ydata[idx[29][0]:idx[29][1]],
        Xtest=Xdata[idx[29][1]:idx[29][2]],
        ytest=ydata[idx[29][1]:idx[29][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_29: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_29.append(pred_t)
pred_NN3.append(sum(pred_29)/len(seed))
del pred_29
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.023593, test loss: 1792.753174
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.023615, test loss: 1802.756836
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.023540, test loss: 1791.635620
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.023801, test loss: 1802.622314
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.023768, test loss: 1810.689575
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.023624, test loss: 1799.731934
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.024022, test loss: 1819.273193
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.023538, test loss: 1783.384521
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.023655, test loss: 1806.453003
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.023731, test loss: 1802.700562
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.023612, test loss: 1790.447998
lambda: 1e

75

In [22]:
#NN3_split_30
pred_30 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[30][0]],
        ytrain=ydata[:idx[30][0]],
        Xval=Xdata[idx[30][0]:idx[30][1]],
        yval=ydata[idx[30][0]:idx[30][1]],
        Xtest=Xdata[idx[30][1]:idx[30][2]],
        ytest=ydata[idx[30][1]:idx[30][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_30: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_30.append(pred_t)
pred_NN3.append(sum(pred_30)/len(seed))
del pred_30
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.024109, test loss: 1317.159912
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.024160, test loss: 1331.804565
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.024109, test loss: 1332.849731
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.024129, test loss: 1328.443604
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.024033, test loss: 1315.888428
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.024165, test loss: 1334.682617
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.024042, test loss: 1320.154541
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.024089, test loss: 1319.988647
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.024192, test loss: 1327.471191
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.024157, test loss: 1330.122192
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.024030, test loss: 1322.744629
lambda: 0.

0

In [23]:
#NN3_split_31
pred_31 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[31][0]],
        ytrain=ydata[:idx[31][0]],
        Xval=Xdata[idx[31][0]:idx[31][1]],
        yval=ydata[idx[31][0]:idx[31][1]],
        Xtest=Xdata[idx[31][1]:idx[31][2]],
        ytest=ydata[idx[31][1]:idx[31][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_31: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_31.append(pred_t)
pred_NN3.append(sum(pred_31)/len(seed))
del pred_31
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.024498, test loss: 1554.932495
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.024630, test loss: 1582.983765
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.024492, test loss: 1551.949463
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.024633, test loss: 1597.772461
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.024622, test loss: 1590.129883
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.025200, test loss: 1676.987183
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.024509, test loss: 1561.592285
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.024534, test loss: 1573.496948
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.024685, test loss: 1551.157104
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.024569, test loss: 1565.668701
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.024522, test loss: 1557.593506
lambda: 

0

In [24]:
import pickle
with open('pred_NN3.pkl', 'wb') as f:
    pickle.dump(pred_NN3, f)

In [25]:
#NN3_split_32
pred_32 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[32][0]],
        ytrain=ydata[:idx[32][0]],
        Xval=Xdata[idx[32][0]:idx[32][1]],
        yval=ydata[idx[32][0]:idx[32][1]],
        Xtest=Xdata[idx[32][1]:idx[32][2]],
        ytest=ydata[idx[32][1]:idx[32][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_32: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_32.append(pred_t)
pred_NN3.append(sum(pred_32)/len(seed))
with open('pred_NN3.pkl', 'wb') as f:
    pickle.dump(pred_NN3, f)
del pred_32
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.025518, test loss: 2327.040039
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.025346, test loss: 2322.641846
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.025548, test loss: 2343.441406
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.025356, test loss: 2319.950439
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.025347, test loss: 2319.915283
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.025328, test loss: 2307.361328
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.025330, test loss: 2312.365479
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.025345, test loss: 2321.604492
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.025275, test loss: 2312.687012
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.025364, test loss: 2316.577637
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.025322, test loss: 2320.272461
lambda: 1e

57

In [26]:
#NN3_split_33
pred_33 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[33][0]],
        ytrain=ydata[:idx[33][0]],
        Xval=Xdata[idx[33][0]:idx[33][1]],
        yval=ydata[idx[33][0]:idx[33][1]],
        Xtest=Xdata[idx[33][1]:idx[33][2]],
        ytest=ydata[idx[33][1]:idx[33][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_33: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_33.append(pred_t)
pred_NN3.append(sum(pred_33)/len(seed))
with open('pred_NN3.pkl', 'wb') as f:
    pickle.dump(pred_NN3, f)
del pred_33
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.026942, test loss: 4690.520508
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.027048, test loss: 4653.049316
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.026984, test loss: 4651.330078
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.026988, test loss: 4653.580566
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.026990, test loss: 4682.578125
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.027238, test loss: 4640.526367
lambda: 0.0001, learning rate: 0.01, epoch: 0, val loss: 0.027047, test loss: 4691.927734
lambda: 0.0001, learning rate: 0.01, epoch: 1, val loss: 0.027035, test loss: 4646.285156
lambda: 0.0001, learning rate: 0.01, epoch: 2, val loss: 0.027619, test loss: 4663.634766
lambda: 0.0001, learning rate: 0.01, epoch: 3, val loss: 0.027157, test loss: 4707.670898
lambda: 0.0001, learning rate: 0.01, epoch: 4, val loss: 0.027100, test loss: 4702.025879
lambda

76

In [27]:
#NN3_split_34
pred_34 = []
for s in seed:
    print(f'seed: {s}')
    minvalloss, best_lambda, best_lr, model_dict, pred_t, sse = train_loop(
        Xtrain=Xdata[:idx[34][0]],
        ytrain=ydata[:idx[34][0]],
        Xval=Xdata[idx[34][0]:idx[34][1]],
        yval=ydata[idx[34][0]:idx[34][1]],
        Xtest=Xdata[idx[34][1]:idx[34][2]],
        ytest=ydata[idx[34][1]:idx[34][2]],
        reg_param=reg_param,
        learningrate=learning_rate,
        n_batchsize=10000,
        model=model_NN3,
        max_epochs=100,
        n_patience=5,
        seed=s
    )
    print(
        f'split_34: lambda = {best_lambda}, learning rate = {best_lr}, min validation loss = {minvalloss:.6f}, test loss: {sse:.6f}')
    pred_34.append(pred_t)
pred_NN3.append(sum(pred_34)/len(seed))
with open('pred_NN3.pkl', 'wb') as f:
    pickle.dump(pred_NN3, f)
del pred_34
gc.collect()

seed: 555
lambda: 1e-05, learning rate: 0.01, epoch: 0, val loss: 0.029354, test loss: 3318.768311
lambda: 1e-05, learning rate: 0.01, epoch: 1, val loss: 0.029477, test loss: 3343.394287
lambda: 1e-05, learning rate: 0.01, epoch: 2, val loss: 0.029341, test loss: 3321.574951
lambda: 1e-05, learning rate: 0.01, epoch: 3, val loss: 0.029352, test loss: 3317.064941
lambda: 1e-05, learning rate: 0.01, epoch: 4, val loss: 0.029311, test loss: 3334.227051
lambda: 1e-05, learning rate: 0.01, epoch: 5, val loss: 0.029279, test loss: 3318.873047
lambda: 1e-05, learning rate: 0.01, epoch: 6, val loss: 0.029378, test loss: 3331.113281
lambda: 1e-05, learning rate: 0.01, epoch: 7, val loss: 0.029437, test loss: 3355.994629
lambda: 1e-05, learning rate: 0.01, epoch: 8, val loss: 0.029311, test loss: 3324.079102
lambda: 1e-05, learning rate: 0.01, epoch: 9, val loss: 0.029553, test loss: 3346.572510
lambda: 1e-05, learning rate: 0.01, epoch: 10, val loss: 0.029357, test loss: 3327.264160
lambda: 0.

95

In [15]:
# Replication period R_squared
y_pred_rep = torch.cat(pred_NN3[:30], dim=0).to(torch.device('cpu'))
R_squared_rep = 1 - (y_true_rep - y_pred_rep).pow(2).sum() / (y_true_rep).pow(2).sum()
R_squared_top1000_rep = 1 - (y_true_rep[t_rep] - y_pred_rep[t_rep]).pow(2).sum() / (y_true_rep[t_rep]).pow(2).sum()
R_squared_bot1000_rep = 1 - (y_true_rep[b_rep] - y_pred_rep[b_rep]).pow(2).sum() / (y_true_rep[b_rep]).pow(2).sum()

# Extended period R_squared
y_pred = torch.cat(pred_NN3, dim=0).to(torch.device('cpu'))
R_squared = 1 - (y_true - y_pred).pow(2).sum() / (y_true).pow(2).sum()
R_squared_top1000 = 1 - (y_true[t] - y_pred[t]).pow(2).sum() / (y_true[t]).pow(2).sum()
R_squared_bot1000 = 1 - (y_true[b] - y_pred[b]).pow(2).sum() / (y_true[b]).pow(2).sum()

Table_1_Monthly_outofsample_stocklevel_prediction = pd.DataFrame(
    data={
    'Replicated (1987-2016)': [R_squared_rep.item()*100, R_squared_top1000_rep.item()*100, R_squared_bot1000_rep.item()*100],
    'Original (1987-2016)': [0.40, 0.70, 0.45],
    'Rep. quality': ['good','not good','good'],
    'Extended (1987-2021)': [R_squared.item()*100, R_squared_top1000.item()*100, R_squared_bot1000.item()*100]
}, index=['R^2_all', 'R^2_top1000', 'R^2_bot1000']
)

print(f'Table 1: Monthly out-of-sample stock prediction R_squared (%)')
print(Table_1_Monthly_outofsample_stocklevel_prediction)

Table 1: Monthly out-of-sample stock prediction R_squared (%)
             Replicated (1987-2016)  Original (1987-2016) Rep. quality  \
R^2_all                    0.383341                  0.40         good   
R^2_top1000                0.419015                  0.70     not good   
R^2_bot1000                0.716943                  0.45         good   

             Extended (1987-2021)  
R^2_all                  0.365525  
R^2_top1000              0.507140  
R^2_bot1000              0.644702  


In [14]:
#Save out-of-sample observed and predicted excess returns
torch.save(y_pred, 'y_pred_NN3.pth')
torch.save(y_pred_rep, 'y_pred_rep_NN3.pth')
torch.save(y_true, 'y_true.pth')
torch.save(y_true_rep, 'y_true_rep.pth')